# Workshop Geomatikkdagene 2026

Last ned følgende filer og pakk dem ut i data-mappen:

- Fra [Basisdata_0000_Norge_25833_N1000Kartdata_FGDB.zip](https://nedlasting.geonorge.no/geonorge/Basisdata/N2000Kartdata/FGDB/Basisdata_0000_Norge_25833_N1000Kartdata_FGDB.zip) til `data/Basisdata_0000_Norge_25833_N1000Kartdata_FGDB.gdb`.
- Fra [Befolkning_0000_Norge_25833_KirkebyggForenklet_FGDB.zip](https://nedlasting.geonorge.no/geonorge/Befolkning/KirkebyggForenklet/FGDB/Befolkning_0000_Norge_25833_KirkebyggForenklet_FGDB.zip) til `data/Befolkning_0000_Norge_25833_KirkebyggForenklet_FGDB.gdb`.
  

In [ ]:
%pip install folium geopandas

In [1]:
import folium
import geopandas as gpd
import numpy as np

In [3]:
# Last inn data
kommuner = gpd.read_file(
    filename="data/Basisdata_0000_Norge_25833_N1000Kartdata_FGDB.gdb",
    layer="N1000_AdministrativeOmråder_omrade",
)

# Transformer til EPSG:4326
kommuner = kommuner.to_crs(4326)

# Gjør om alle geometrier til Polygon
kommuner = kommuner.explode()

# Forhåndsvisning
display(kommuner.head())

# Lagre til Parquet
kommuner.to_parquet("data/kommuner_n1000.parquet")

,objtype,kommunenavn,kommunenummer,fylkesnavn,fylkesnummer,oppdateringsdato,SHAPE_Length,SHAPE_Area,geometry
0,Kommune,Marker,3122,Østfold,31,2025-12-17 00:00:00+00:00,113162.542333,4.123184e+08,"POLYGON ((11.62989 59.67682, 11.6421 59.67538,..."
1,Kommune,Søndre Land,3447,Innlandet,34,2025-12-17 00:00:00+00:00,130821.413993,7.292558e+08,"POLYGON ((10.30183 60.86781, 10.3311 60.85787,..."
2,Kommune,Haugesund,1106,Rogaland,11,2025-12-17 00:00:00+00:00,117140.575463,3.682784e+08,"POLYGON ((4.97689 59.49613, 5.18741 59.51548, ..."
3,Kommune,Øyer,3440,Innlandet,34,2025-12-17 00:00:00+00:00,111882.458161,6.405435e+08,"POLYGON ((10.7534 61.40472, 10.77021 61.38063,..."
4,Kommune,Frøya,5014,Trøndelag,50,2025-12-17 00:00:00+00:00,310262.579864,5.299279e+09,"POLYGON ((9.67105 64.41509, 9.70015 64.39592, ..."


In [4]:
# Last inn data
kirkebygg = gpd.read_file(
    columns=["objtype", "bygningsnavn", "lokalid", "godkjentkapasitet", "hovedmateriale"],
    filename="data/Befolkning_0000_Norge_25833_KirkebyggForenklet_FGDB.gdb",
    layer="kirkebyggenkel_kirkebygg",
)

# Transformer til EPSG:4326
kirkebygg = kirkebygg.to_crs(4326)

# Ta vekk kirken som ligger på Svalbard
kirkebygg = kirkebygg[kirkebygg["lokalid"] != "5f523858-2268-4ea8-a755-262f0eaf9d9d"]

# Gjør 'godkjentkapasitet' om til heltall
kirkebygg["godkjentkapasitet"] = kirkebygg["godkjentkapasitet"].astype("Int32")

# Finn alle kirker som mangler 'godkjentkapasitet'
mangler_kapasitet = kirkebygg["godkjentkapasitet"].isna()

# Lag en liste med kapasiteter som vi kan bruke der det mangler
kapasiteter = [250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350]

# Legg til tilfeldig kapasitet der det mangler
kirkebygg.loc[mangler_kapasitet, "godkjentkapasitet"] = np.random.choice(kapasiteter, mangler_kapasitet.sum())

# Forhåndsvisning
display(kirkebygg.head())

# Lagre til Parquet
kirkebygg.to_parquet("data/kirkebygg_forenklet.parquet")

,bygningsnavn,godkjentkapasitet,hovedmateriale,lokalid,objtype,geometry
0,Slettebakken kirke,250,mur,f73c5edd-bc94-470e-bed8-b5ad08c3d1b2,Kirkebygg,POINT (5.35694 60.35285)
1,Sælen kirke,250,mur,8c993bd4-2517-4018-afe6-9e709223c931,Kirkebygg,POINT (5.26968 60.34057)
2,Fyllingsdalen kirke,340,mur,f23c9cea-0679-4111-8c22-ef5f8499e278,Kirkebygg,POINT (5.29645 60.35299)
3,Bønes kirke,310,tre,641f5544-7c94-4387-ac98-02ab4b4e7abf,Kirkebygg,POINT (5.30162 60.32992)
4,Storetveit kirke,330,mur,a3ad918d-c034-4b23-a84b-1010e953ace8,Kirkebygg,POINT (5.345 60.35159)


In [13]:
%pip install fsspec

Note: you may need to restart the kernel to use updated packages.


In [21]:
##kommune_original = kommuner[kommuner["kommunenummer"] == "3305"][["kommunenummer", "kommunenavn", "geometry"]].copy()

# read parquet file with geopandas and filter for a specific kommune directly in the query
uri = "az://skygeo/workshopdata/kommuner_n1000.parquet"
account_name = "kartaistorage"

kommune_original = gpd.read_parquet(
    uri, 
    storage_options={"account_name": account_name}
)

kommune_original = kommune_original[kommune_original["kommunenummer"] == "3305"][["kommunenummer", "kommunenavn", "geometry"]].copy()

kommune_buffer = kommune_original.copy()

kommune_buffer = kommune_buffer.to_crs(3857)

kommune_buffer["geometry"] = kommune_buffer.buffer(15_000)

kommune_buffer = kommune_buffer.to_crs(4326)

In [22]:
kirker_i_original = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_original)

kirker_i_original_antall = len(kirker_i_original)

kirker_i_original_kapasitet = kirker_i_original["godkjentkapasitet"].sum()

display(f"Antall kirker i kommune er {kirker_i_original_antall} og rommer {kirker_i_original_kapasitet} personer.")

kirker_i_buffer = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_buffer)

kirker_i_buffer_antall = len(kirker_i_buffer)

kirker_i_buffer_kapasitet = kirker_i_buffer["godkjentkapasitet"].sum()

display(f"Antall kirker i buffer er {kirker_i_buffer_antall} og rommer {kirker_i_buffer_kapasitet} personer.")

'Antall kirker i kommune er 14 og rommer 4360 personer.'

'Antall kirker i buffer er 26 og rommer 7740 personer.'

In [8]:
xmin, ymin, xmax, ymax = kommune_buffer.total_bounds.tolist()

m = folium.Map(
    tiles="OpenStreetMap",
)

m.fit_bounds([[ymin, xmin], [ymax, xmax]])

folium.GeoJson(
    data=kommune_original,
    color="blue",
    fill_opacity=0.0,
    weight=2.0,
).add_to(m)

folium.GeoJson(
    data=kommune_buffer,
    color="red",
    fill_opacity=0.0,
    weight=1.0,
    dash_array=[10.0, 5.0],
).add_to(m)

folium.GeoJson(
    data=kirker_i_buffer,
    tooltip=folium.GeoJsonTooltip(fields=["bygningsnavn"]),
    marker=folium.CircleMarker(
        color="black",
        fill_color="white",
        fill_opacity=1.0,
        radius=5.0,
        weight=1,
    ),
).add_to(m)

m